In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SEPARAR DATASET EN ACTIVOS / ELIMINADOS

In [2]:
# Importando la biblioteca pandas para manipulación y análisis de datos
import pandas as pd
business_payments = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/merged_inner.csv')

In [ ]:
business_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21057 entries, 0 to 21056
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          21057 non-null  int64  
 1   cash_request_id             21057 non-null  int64  
 2   type                        21057 non-null  object 
 3   status                      21057 non-null  object 
 4   category                    2196 non-null   object 
 5   total_amount                21057 non-null  float64
 6   reason                      21057 non-null  object 
 7   created_at                  21057 non-null  object 
 8   updated_at                  21057 non-null  object 
 9   paid_at                     15438 non-null  object 
 10  from_date                   6749 non-null   object 
 11  to_date                     6512 non-null   object 
 12  charge_moment               21057 non-null  object 
 13  amount                      210

In [ ]:
import pandas as pd

# Cargar el archivo CSV original
file_path = 'drive/MyDrive/ColabNotebooks/Business_Payments/merged_inner.csv'  # Cambia esta ruta según corresponda
data = pd.read_csv(file_path)

# Separar los datos en usuarios activos y eliminados
usuarios_activos = data[data['user_id'].notna()]
usuarios_eliminados = data[data['deleted_account_id'].notna()]

# Guardar los datos en archivos CSV separados
usuarios_activos.to_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos.csv', index=False)
usuarios_eliminados.to_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados.csv.csv', index=False)

print("Archivos generados: 'usuarios_activos.csv' y 'usuarios_eliminados.csv'")


Archivos generados: 'usuarios_activos.csv' y 'usuarios_eliminados.csv'


--------------------------------------------------------------------------------------------------------

INGENIERIA DE *DATOS*



In [14]:
import pandas as pd

In [20]:
usuarios_activos = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos.csv')
usuarios_eliminados = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados.csv')

In [21]:
# Eliminar columnas no necesarias
usuarios_activos = usuarios_activos.drop(columns=['deleted_account_id'], errors='ignore')
usuarios_eliminados = usuarios_eliminados.drop(columns=['user_id'], errors='ignore')

COLUMNA TOTAL SOLICITUDES

In [22]:
import pandas as pd

# Contar solicitudes por usuario en usuarios activos
solicitudes_activos = usuarios_activos["user_id"].value_counts().reset_index()
solicitudes_activos.columns = ["user_id", "total_solicitudes_usuario"]

# Unir la información al dataset original
usuarios_activos = usuarios_activos.merge(solicitudes_activos, on="user_id", how="left")

# Contar solicitudes por usuario en usuarios eliminados
solicitudes_eliminados = usuarios_eliminados["deleted_account_id"].value_counts().reset_index()
solicitudes_eliminados.columns = ["deleted_account_id", "total_solicitudes_usuario"]

# Unir la información al dataset original
usuarios_eliminados = usuarios_eliminados.merge(solicitudes_eliminados, on="deleted_account_id", how="left")

COLUMNA TOTAL OPERACIONES CANCELADAS O RECHAZADAS

In [24]:
import pandas as pd

# Contar operaciones canceladas o rechazadas en usuarios activos
canceladas_rechazadas_activos = usuarios_activos[
    usuarios_activos["status"].isin(["cancelled", "rejected"])
]["user_id"].value_counts().reset_index()
canceladas_rechazadas_activos.columns = ["user_id", "total_operaciones_canceladas_rechazadas"]

# Unir la información al dataset original
usuarios_activos = usuarios_activos.merge(canceladas_rechazadas_activos, on="user_id", how="left")
usuarios_activos["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)

# Contar operaciones canceladas o rechazadas en usuarios eliminados
canceladas_rechazadas_eliminados = usuarios_eliminados[
    usuarios_eliminados["status"].isin(["cancelled", "rejected"])
]["deleted_account_id"].value_counts().reset_index()
canceladas_rechazadas_eliminados.columns = ["deleted_account_id", "total_operaciones_canceladas_rechazadas"]

# Unir la información al dataset original
usuarios_eliminados = usuarios_eliminados.merge(canceladas_rechazadas_eliminados, on="deleted_account_id", how="left")
usuarios_eliminados["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)

   user_id  total_operaciones_canceladas_rechazadas
0  35661.0                                      2.0
1  16158.0                                      3.0
2  81575.0                                      0.0
3  94393.0                                      0.0
4  90386.0                                      0.0
   deleted_account_id  total_operaciones_canceladas_rechazadas
0             19005.0                                      0.0
1             29610.0                                      0.0
2             23059.0                                      1.0
3             26653.0                                      4.0
4             23893.0                                      2.0


<ipython-input-24-66d1d0346ff5>:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)
<ipython-input-24-66d1d0346ff5>:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].m

COLUMNA MES DE LA SOLICITUD

In [25]:
# Convertir created_at a zona horaria UTC y extraer el mes de la solicitud
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")

usuarios_activos["mes_solicitud"] = usuarios_activos["created_at"].dt.month
usuarios_eliminados["mes_solicitud"] = usuarios_eliminados["created_at"].dt.month

# Verificar resultado
print(usuarios_activos[["user_id", "created_at", "mes_solicitud"]].head())
print(usuarios_eliminados[["deleted_account_id", "created_at", "mes_solicitud"]].head())


   user_id                       created_at  mes_solicitud
0  35661.0 2020-09-07 10:47:27.423150+00:00              9
1  16158.0 2020-09-09 20:51:17.998653+00:00              9
2  81575.0 2020-10-23 10:10:58.352972+00:00             10
3  94393.0 2020-10-31 15:46:53.643958+00:00             10
4  90386.0 2020-10-24 12:22:27.666102+00:00             10
   deleted_account_id                       created_at  mes_solicitud
0             19005.0 2020-10-06 08:20:17.170432+00:00             10
1             29610.0 2020-10-24 10:22:47.111527+00:00             10
2             23059.0 2020-10-16 23:48:55.379796+00:00             10
3             26653.0 2020-11-01 23:01:27.934752+00:00             11
4             23893.0 2020-10-06 06:34:48.209743+00:00             10


In [32]:
usuarios_activos["semana_solicitud"] = usuarios_activos["created_at"].dt.isocalendar().week
usuarios_eliminados["semana_solicitud"] = usuarios_eliminados["created_at"].dt.isocalendar().week


In [33]:
usuarios_activos["dia_semana_solicitud"] = usuarios_activos["created_at"].dt.dayofweek
usuarios_eliminados["dia_semana_solicitud"] = usuarios_eliminados["created_at"].dt.dayofweek


In [34]:
usuarios_activos["hora_solicitud"] = usuarios_activos["created_at"].dt.hour
usuarios_eliminados["hora_solicitud"] = usuarios_eliminados["created_at"].dt.hour


In [26]:
# Evitar división por cero
usuarios_activos["tasa_rechazo"] = usuarios_activos["total_operaciones_canceladas_rechazadas"] / usuarios_activos["total_solicitudes_usuario"]
usuarios_eliminados["tasa_rechazo"] = usuarios_eliminados["total_operaciones_canceladas_rechazadas"] / usuarios_eliminados["total_solicitudes_usuario"]

# Reemplazar NaN e infinitos por 0
usuarios_activos["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)
usuarios_eliminados["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "tasa_rechazo"]].head())
print(usuarios_eliminados[["deleted_account_id", "tasa_rechazo"]].head())


   user_id  tasa_rechazo
0  35661.0           1.0
1  16158.0           0.5
2  81575.0           0.0
3  94393.0           0.0
4  90386.0           0.0
   deleted_account_id  tasa_rechazo
0             19005.0      0.000000
1             29610.0      0.000000
2             23059.0      0.250000
3             26653.0      0.666667
4             23893.0      0.500000


<ipython-input-26-702cbbddd062>:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)
<ipython-input-26-702cbbddd062>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col]

In [27]:
# Calcular la diferencia de tiempo entre solicitudes por usuario
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True)
usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True)

# Calcular diferencia de días entre solicitudes y sacar el promedio por usuario
intervalo_activos = usuarios_activos.sort_values(["user_id", "created_at"]).groupby("user_id")["created_at"].diff().dt.days
intervalo_eliminados = usuarios_eliminados.sort_values(["deleted_account_id", "created_at"]).groupby("deleted_account_id")["created_at"].diff().dt.days

usuarios_activos["intervalo_promedio_solicitudes"] = usuarios_activos["user_id"].map(intervalo_activos.groupby(usuarios_activos["user_id"]).mean())
usuarios_eliminados["intervalo_promedio_solicitudes"] = usuarios_eliminados["deleted_account_id"].map(intervalo_eliminados.groupby(usuarios_eliminados["deleted_account_id"]).mean())

# Rellenar NaN con un valor alto (por ejemplo, 9999 para usuarios con una sola solicitud)
usuarios_activos["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)
usuarios_eliminados["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "intervalo_promedio_solicitudes"]].head())
print(usuarios_eliminados[["deleted_account_id", "intervalo_promedio_solicitudes"]].head())


   user_id  intervalo_promedio_solicitudes
0  35661.0                            34.0
1  16158.0                            21.0
2  81575.0                          9999.0
3  94393.0                          9999.0
4  90386.0                          9999.0
   deleted_account_id  intervalo_promedio_solicitudes
0             19005.0                       28.750000
1             29610.0                     9999.000000
2             23059.0                       21.666667
3             26653.0                       10.400000
4             23893.0                       17.000000


<ipython-input-27-2f376469ec71>:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)
<ipython-input-27-2f376469ec71>:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

In [35]:
# Asegurar que las fechas sean tipo datetime
usuarios_activos["created_at"] = pd.to_datetime(usuarios_activos["created_at"], utc=True, errors="coerce")
usuarios_activos["paid_at"] = pd.to_datetime(usuarios_activos["paid_at"], utc=True, errors="coerce")

usuarios_eliminados["created_at"] = pd.to_datetime(usuarios_eliminados["created_at"], utc=True, errors="coerce")
usuarios_eliminados["paid_at"] = pd.to_datetime(usuarios_eliminados["paid_at"], utc=True, errors="coerce")

# Calcular la diferencia en días entre pago y creación
usuarios_activos["dias_para_pago"] = (usuarios_activos["paid_at"] - usuarios_activos["created_at"]).dt.days
usuarios_eliminados["dias_para_pago"] = (usuarios_eliminados["paid_at"] - usuarios_eliminados["created_at"]).dt.days

# Definir pagos tardíos (más de 30 días)
usuarios_activos["pago_tardio"] = (usuarios_activos["dias_para_pago"] > 30).astype(int)
usuarios_eliminados["pago_tardio"] = (usuarios_eliminados["dias_para_pago"] > 30).astype(int)

# Calcular la proporción de pagos tardíos por usuario
pago_tardio_ratio_activos = usuarios_activos.groupby("user_id")["pago_tardio"].mean().reset_index()
pago_tardio_ratio_eliminados = usuarios_eliminados.groupby("deleted_account_id")["pago_tardio"].mean().reset_index()

# Renombrar columna
pago_tardio_ratio_activos.columns = ["user_id", "pago_tardio_ratio"]
pago_tardio_ratio_eliminados.columns = ["deleted_account_id", "pago_tardio_ratio"]

# Unir al dataset original
usuarios_activos = usuarios_activos.merge(pago_tardio_ratio_activos, on="user_id", how="left")
usuarios_eliminados = usuarios_eliminados.merge(pago_tardio_ratio_eliminados, on="deleted_account_id", how="left")

# Verificar resultado
print(usuarios_activos[["user_id", "pago_tardio_ratio"]].head())
print(usuarios_eliminados[["deleted_account_id", "pago_tardio_ratio"]].head())



   user_id  pago_tardio_ratio
0  35661.0                1.0
1  16158.0                0.5
2  81575.0                0.0
3  94393.0                0.0
4  90386.0                0.0
   deleted_account_id  pago_tardio_ratio
0             19005.0           0.000000
1             29610.0           0.000000
2             23059.0           0.250000
3             26653.0           0.166667
4             23893.0           0.000000


In [31]:
# Contar cuántas veces se ha modificado una solicitud
usuarios_activos["solicitudes_modificadas"] = usuarios_activos.groupby("user_id")["updated_at"].transform("count") - 1
usuarios_eliminados["solicitudes_modificadas"] = usuarios_eliminados.groupby("deleted_account_id")["updated_at"].transform("count") - 1

# Evitar valores negativos (en caso de usuarios con una sola solicitud)
usuarios_activos["solicitudes_modificadas"] = usuarios_activos["solicitudes_modificadas"].clip(lower=0)
usuarios_eliminados["solicitudes_modificadas"] = usuarios_eliminados["solicitudes_modificadas"].clip(lower=0)

# Verificar resultado
print(usuarios_activos[["user_id", "solicitudes_modificadas"]].head())
print(usuarios_eliminados[["deleted_account_id", "solicitudes_modificadas"]].head())


   user_id  solicitudes_modificadas
0  35661.0                        1
1  16158.0                        5
2  81575.0                        0
3  94393.0                        0
4  90386.0                        0
   deleted_account_id  solicitudes_modificadas
0             19005.0                        4
1             29610.0                        0
2             23059.0                        3
3             26653.0                        5
4             23893.0                        3


In [37]:
# Guardar los archivos actualizados (opcional)
usuarios_activos.to_csv("drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos_actualizado.csv", index=False)
usuarios_eliminados.to_csv("drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados_actualizado.csv", index=False)

In [38]:
usuarios_activos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20151 entries, 0 to 20150
Data columns (total 39 columns):
 #   Column                                   Non-Null Count  Dtype              
---  ------                                   --------------  -----              
 0   id                                       20151 non-null  int64              
 1   cash_request_id                          20151 non-null  int64              
 2   type                                     20151 non-null  object             
 3   status                                   20151 non-null  object             
 4   category                                 2030 non-null   object             
 5   total_amount                             20151 non-null  float64            
 6   reason                                   20151 non-null  object             
 7   created_at                               20151 non-null  datetime64[ns, UTC]
 8   updated_at                               20151 non-null  object   